# 09n — MOL00484 libera in acqua: confronto con lo stato site-relevant associato a EGFR
**PEARL · Windows CUDA · kernel Python (PEARL GPU)**

Follow-up parallelo a 09j, adattato a **MOL00484**. Il sistema libero usa la parametrizzazione autorevole di MOL00484 derivata da 09l (OpenFF Sage 2.2.1 + cariche AM1-BCC) e il sistema acqua/TIP3P pre-solvatato e validato su Mac, poi verificato su Windows/CUDA.

La produzione ufficiale comprende **3 repliche × 10 ns**, precedute per ciascuna replica da minimizzazione, **100 ps NVT** e **200 ps NPT**, a 300 K e 1 bar. Il confronto libero/associato è conformazionale e di idratazione a microstato fisso: **non è un calcolo di ΔG di binding, non determina Kd/Ki, pKa o popolazioni di protonazione e non dimostra affinità o inibizione**.

**Nota di provenienza:** il precedente percorso sperimentale di ricostruzione manuale del sistema su Windows e lo smoke test CUDA sono stati rimossi dal flusso ufficiale. Lo smoke test resta documentato separatamente nell'output `20261028T080153_298689Z_smoke`.


In [5]:
from pathlib import Path
import json, hashlib, time, random, sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import openmm as mm
from openmm import app, unit
import mdtraj as mt
import matplotlib.pyplot as plt

PEARL_ROOT = Path(r"C:\Users\Roberto\PEARL")
ASSETS_09N = PEARL_ROOT / "inputs" / "09n_MOL00484"
SOURCE_09L = PEARL_ROOT / "MOL00484_09l_WINDOWS_1ns"
BOUND_09L = SOURCE_09L / "production_runs" / "20261027T171745_698045Z_seed20261027"
OUTPUT_BASE = PEARL_ROOT / "outputs" / "pipeline_5_09n_mol00484_free_water"

FREE_PDB = ASSETS_09N / "MOL00484_free_water_solvated.pdb"
FREE_SYSTEM_XML = ASSETS_09N / "MOL00484_free_water_physical_system.xml"
FREE_QC = ASSETS_09N / "MOL00484_free_water_system_qc.json"

ATOM_MAP = SOURCE_09L / "tables" / "solute_atom_map.csv"
BOUND_TOPOLOGY = SOURCE_09L / "structures" / "solute_reference_09f.pdb"
BOUND_DCD = BOUND_09L / "trajectories" / "production_solute.dcd"

RUN_MD = True
SEEDS = [20261029, 20261030, 20261031]

NVT_PS = 100.0
NPT_PS = 200.0
PRODUCTION_PS = 10000.0
FRAME_PS = 10.0

DT_PS = 0.002
TEMPERATURE_K = 300.0
PRESSURE_BAR = 1.0
DISCARD_FRACTION = 0.20
SAVE_FULL_DCD = False

PLATFORM = "CUDA"
PLATFORM_PROPERTIES = {"DeviceIndex": "0", "Precision": "mixed"}

def require(ok, msg):
    if not ok:
        raise RuntimeError(msg)

def dump(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")

def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(1024*1024), b""):
            h.update(b)
    return h.hexdigest()

def steps(ps):
    n = round(ps / DT_PS)
    require(n > 0 and abs(n * DT_PS - ps) < 1e-9, "Durata non multipla di 2 fs.")
    return n

print("=== 09n OFFICIAL PRODUCTION PREFLIGHT ===")
print("Python :", sys.version.split()[0])
print("OpenMM :", mm.__version__)
print("MDTraj :", mt.__version__)

for name, path in {
    "free_water_pdb": FREE_PDB,
    "free_water_system": FREE_SYSTEM_XML,
    "free_water_qc": FREE_QC,
    "09l_atom_map": ATOM_MAP,
    "09l_bound_topology": BOUND_TOPOLOGY,
    "09l_bound_dcd": BOUND_DCD,
}.items():
    print(f"{name:20s}: {path.is_file()} | {path}")
    require(path.is_file(), f"Input assente: {path}")

pdb0 = app.PDBFile(str(FREE_PDB))
base_system = mm.XmlSerializer.deserialize(FREE_SYSTEM_XML.read_text(encoding="utf-8"))
qc0 = json.loads(FREE_QC.read_text(encoding="utf-8"))

require(pdb0.topology.getNumAtoms() == 2171, "Numero atomi free-water inatteso.")
require(base_system.getNumParticles() == 2171, "Numero particelle free-water inatteso.")
require(base_system.getNumConstraints() == 2142, "Numero constraints inatteso.")
require(qc0["ligand_atoms"] == 46 and qc0["ligand_bonds"] == 48, "QC ligando inatteso.")
require(qc0["ligand_forcefield"] == "openff-2.2.1", "Force field ligando inatteso.")
require(qc0["water_forcefield"] == "amber14/tip3p.xml", "Force field acqua inatteso.")

atommap = pd.read_csv(ATOM_MAP)
R = atommap.index[atommap["component"].eq("EGFR")].to_numpy(int)
L = atommap.index[atommap["component"].eq("MOL00484")].to_numpy(int)
require(len(R) == 9286 and len(L) == 46, "Atom map 09l inattesa.")

free_top = mt.Topology.from_openmm(pdb0.topology)
ligand_indices = np.arange(46, dtype=int)
ligtop = free_top.subset(ligand_indices)
heavy = np.array([a.index for a in ligtop.atoms if a.element.symbol != "H"], dtype=int)
ligbonds = np.array([[a.index, b.index] for a, b in ligtop.bonds], dtype=int)

require(ligtop.n_atoms == 46, "Topologia ligando inattesa.")
require(len(ligbonds) == 48, "Numero legami ligando inatteso.")

print("Free-water atoms :", pdb0.topology.getNumAtoms())
print("Ligand atoms     :", ligtop.n_atoms)
print("Ligand heavy     :", len(heavy))
print("Ligand bonds     :", len(ligbonds))
print("Replicas         :", len(SEEDS))
print("Production/rep   :", PRODUCTION_PS, "ps")
print("CUDA             :", PLATFORM_PROPERTIES)
print("\n09n OFFICIAL PREFLIGHT: OK")

if not RUN_MD:
    print("\nSAFE STOP: RUN_MD=False")


=== 09n OFFICIAL PRODUCTION PREFLIGHT ===
Python : 3.11.16
OpenMM : 8.6.1
MDTraj : 1.11.1
free_water_pdb      : True | C:\Users\Roberto\PEARL\inputs\09n_MOL00484\MOL00484_free_water_solvated.pdb
free_water_system   : True | C:\Users\Roberto\PEARL\inputs\09n_MOL00484\MOL00484_free_water_physical_system.xml
free_water_qc       : True | C:\Users\Roberto\PEARL\inputs\09n_MOL00484\MOL00484_free_water_system_qc.json
09l_atom_map        : True | C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\tables\solute_atom_map.csv
09l_bound_topology  : True | C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\structures\solute_reference_09f.pdb
09l_bound_dcd       : True | C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\production_runs\20261027T171745_698045Z_seed20261027\trajectories\production_solute.dcd
Free-water atoms : 2171
Ligand atoms     : 46
Ligand heavy     : 25
Ligand bonds     : 48
Replicas         : 3
Production/rep   : 10000.0 ps
CUDA             : {'DeviceIndex': '0', 'Precision': 'mixed'}



## 1. Produzione ufficiale free-water

Ogni replica parte dallo stesso sistema pre-solvatato e validato, ma usa un seed indipendente per velocità, Langevin integrator e barostato. Per ogni replica: minimizzazione → 100 ps NVT → 200 ps NPT → 10 ns di produzione. Vengono salvati il DCD del solo ligando, log termodinamico, idratazione geometrica e checkpoint. Il DCD completo del solvente resta disabilitato per default.


In [ ]:
# ============================================================
# OFFICIAL PRODUCTION CODE — ALREADY COMPLETED
# DO NOT RERUN UNLESS A NEW INDEPENDENT PRODUCTION IS INTENDED
# Official run: 20261028T080940_432915Z_production
# ============================================================
require(RUN_MD, "RUN_MD=False: produzione non autorizzata.")

RUN = OUTPUT_BASE / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ_production")
RUN.mkdir(parents=True, exist_ok=False)

provenance = {
    "candidate": "MOL00484",
    "stage": "09n",
    "mode": "official_free_water_production",
    "status": "initialized",
    "seeds": SEEDS,
    "nvt_ps": NVT_PS,
    "npt_ps": NPT_PS,
    "production_ps_per_replica": PRODUCTION_PS,
    "frame_ps": FRAME_PS,
    "temperature_K": TEMPERATURE_K,
    "pressure_bar": PRESSURE_BAR,
    "dt_ps": DT_PS,
    "microstate": "neutral_same_as_authoritative_09l_parameterization",
    "ligand_forcefield": "OpenFF Sage 2.2.1",
    "ligand_charges": "AM1-BCC inherited from authoritative 09l JSON",
    "water_forcefield": "amber14/tip3p.xml",
    "free_system_sha256": sha(FREE_SYSTEM_XML),
    "free_pdb_sha256": sha(FREE_PDB),
    "note": "Uses validated pre-solvated system; obsolete manual Windows force-merging path disabled."
}
dump(RUN / "provenance.json", provenance)

def save_state(sim, stem):
    st = sim.context.getState(
        getPositions=True, getVelocities=True, getEnergy=True,
        getParameters=True, enforcePeriodicBox=True
    )
    pe = st.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
    xyz = st.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
    require(np.isfinite(pe), "Energia non finita.")
    require(np.isfinite(xyz).all(), "Coordinate non finite.")
    stem.with_suffix(".xml").write_text(mm.XmlSerializer.serialize(st), encoding="utf-8")
    sim.saveCheckpoint(str(stem.with_suffix(".chk")))
    return st

def state_reporter(path):
    return app.StateDataReporter(
        str(path), steps(FRAME_PS),
        step=True, time=True, potentialEnergy=True, kineticEnergy=True,
        temperature=True, volume=True, density=True, speed=True,
        separator=","
    )

replica_dirs = []
provenance["status"] = "running"
dump(RUN / "provenance.json", provenance)

try:
    for rep, seed in enumerate(SEEDS, 1):
        folder = RUN / f"replica_{rep:02d}_seed{seed}"
        folder.mkdir()
        replica_dirs.append(folder)

        random.seed(seed)
        np.random.seed(seed)

        # Fresh deserialize: every replica gets an independent System object.
        system = mm.XmlSerializer.deserialize(FREE_SYSTEM_XML.read_text(encoding="utf-8"))
        pdb = app.PDBFile(str(FREE_PDB))

        barostat = mm.MonteCarloBarostat(
            PRESSURE_BAR * unit.bar,
            TEMPERATURE_K * unit.kelvin,
            25
        )
        barostat.setRandomNumberSeed(seed + 100)
        barostat.setFrequency(0)
        system.addForce(barostat)

        integ = mm.LangevinMiddleIntegrator(
            TEMPERATURE_K * unit.kelvin,
            1.0 / unit.picosecond,
            DT_PS * unit.picosecond
        )
        integ.setRandomNumberSeed(seed)
        integ.setConstraintTolerance(1e-6)

        sim = app.Simulation(
            pdb.topology, system, integ,
            mm.Platform.getPlatformByName(PLATFORM),
            PLATFORM_PROPERTIES
        )
        sim.context.setPositions(pdb.positions)

        print(f"\nReplica {rep}/{len(SEEDS)} | seed {seed}", flush=True)
        print("Minimizzazione...", flush=True)
        sim.minimizeEnergy(
            tolerance=10.0 * unit.kilojoule_per_mole / unit.nanometer,
            maxIterations=5000
        )
        save_state(sim, folder / "minimized")

        sim.context.setVelocitiesToTemperature(TEMPERATURE_K * unit.kelvin, seed)

        sim.reporters = [state_reporter(folder / "nvt.csv")]
        print("NVT 100 ps...", flush=True)
        sim.step(steps(NVT_PS))
        sim.reporters = []

        barostat.setFrequency(25)
        sim.context.reinitialize(preserveState=True)

        sim.reporters = [state_reporter(folder / "npt.csv")]
        print("NPT 200 ps...", flush=True)
        sim.step(steps(NPT_PS))
        sim.reporters = []
        save_state(sim, folder / "equilibrated")

        sim.context.setTime(0 * unit.picosecond)
        sim.currentStep = 0

        (folder / "production_system.xml").write_text(
            mm.XmlSerializer.serialize(system), encoding="utf-8"
        )
        (folder / "production_integrator.xml").write_text(
            mm.XmlSerializer.serialize(integ), encoding="utf-8"
        )

        fulltop = mt.Topology.from_openmm(pdb.topology)
        waterO = np.array([
            a.index for a in fulltop.atoms
            if a.residue.is_water and a.element.symbol == "O"
        ], dtype=int)
        require(len(waterO) > 0, "Acque assenti.")

        sim.reporters = [
            app.DCDReporter(
                str(folder / "ligand.dcd"),
                steps(FRAME_PS),
                atomSubset=list(range(46)),
                enforcePeriodicBox=False
            ),
            state_reporter(folder / "production.csv"),
            app.CheckpointReporter(str(folder / "latest.chk"), steps(FRAME_PS) * 10),
        ]
        if SAVE_FULL_DCD:
            sim.reporters.append(
                app.DCDReporter(str(folder / "full.dcd"), steps(FRAME_PS), enforcePeriodicBox=False)
            )

        nframes = steps(PRODUCTION_PS) // steps(FRAME_PS)
        hydration = []
        t0 = time.perf_counter()

        for frame in range(1, nframes + 1):
            sim.step(steps(FRAME_PS))

            st = sim.context.getState(getPositions=True)
            xyz = st.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
            box = st.getPeriodicBoxVectors(asNumpy=True).value_in_unit(unit.nanometer)

            require(np.isfinite(xyz).all(), "Coordinate non finite durante produzione.")
            require(np.allclose(box, np.diag(np.diag(box)), atol=1e-6), "Box non ortorombico.")

            lengths = np.diag(box)
            lig_xyz = xyz[:46][heavy]
            delta = xyz[waterO, None, :] - lig_xyz[None, :, :]
            delta -= lengths * np.round(delta / lengths)
            nwater = int((np.linalg.norm(delta, axis=2).min(axis=1) <= 0.35).sum())

            hydration.append({
                "time_ps": frame * FRAME_PS,
                "nearby_water_oxygens_3p5A": nwater
            })

            if frame % max(1, nframes // 20) == 0 or frame == nframes:
                elapsed = time.perf_counter() - t0
                speed = (frame * FRAME_PS / 1000.0) * 86400.0 / elapsed
                remaining_min = ((nframes - frame) / frame * elapsed / 60.0) if frame else float("nan")
                pd.DataFrame(hydration).to_csv(folder / "hydration.csv", index=False)
                print(
                    f"Replica {rep}: {100*frame/nframes:.0f}% | "
                    f"{frame*FRAME_PS/1000:.2f} ns | {speed:.1f} ns/giorno | "
                    f"restano ~{remaining_min:.1f} min",
                    flush=True
                )

        save_state(sim, folder / "final")
        sim.reporters = []

        dump(folder / "completed.json", {
            "seed": seed,
            "production_ps": PRODUCTION_PS,
            "frames": nframes,
            "status": "completed"
        })

        del sim, integ, system

except Exception as exc:
    provenance.update(status="failed", error=repr(exc))
    dump(RUN / "provenance.json", provenance)
    raise

print("\n09n OFFICIAL FREE-WATER PRODUCTION: COMPLETED")
print("Output:", RUN)


In [6]:
# ============================================================
# 09n — RECOVER EXISTING OFFICIAL PRODUCTION
# Do NOT rerun MD
# ============================================================

RUN = Path(
    r"C:\Users\Roberto\PEARL\outputs"
    r"\pipeline_5_09n_mol00484_free_water"
    r"\20261028T080940_432915Z_production"
)

require(RUN.is_dir(), f"Run ufficiale non trovato: {RUN}")

replica_dirs = sorted(RUN.glob("replica_*"))

print("=== 09n RECOVER OFFICIAL PRODUCTION ===")
print("RUN:", RUN)
print("Repliche trovate:", len(replica_dirs))

require(len(replica_dirs) == 3, "Attese esattamente 3 repliche.")

for folder in replica_dirs:
    ligand_dcd = folder / "ligand.dcd"
    production_csv = folder / "production.csv"
    hydration_csv = folder / "hydration.csv"
    completed = folder / "completed.json"

    print(f"\n{folder.name}")
    print("  ligand.dcd     :", ligand_dcd.is_file())
    print("  production.csv :", production_csv.is_file())
    print("  hydration.csv  :", hydration_csv.is_file())
    print("  completed.json :", completed.is_file())

    require(ligand_dcd.is_file(), f"ligand.dcd mancante in {folder}")
    require(production_csv.is_file(), f"production.csv mancante in {folder}")
    require(hydration_csv.is_file(), f"hydration.csv mancante in {folder}")
    require(completed.is_file(), f"completed.json mancante in {folder}")

    info = json.loads(completed.read_text(encoding="utf-8"))

    print("  seed           :", info["seed"])
    print("  production_ps  :", info["production_ps"])
    print("  frames         :", info["frames"])
    print("  status         :", info["status"])

    require(info["production_ps"] == 10000.0, "Produzione non pari a 10 ns.")
    require(info["frames"] == 1000, "Numero frame inatteso.")
    require(info["status"] == "completed", "Replica non completata.")

print("\nDO NOT RERUN MD.")
print("09n OFFICIAL PRODUCTION RECOVERY: OK")

=== 09n RECOVER OFFICIAL PRODUCTION ===
RUN: C:\Users\Roberto\PEARL\outputs\pipeline_5_09n_mol00484_free_water\20261028T080940_432915Z_production
Repliche trovate: 3

replica_01_seed20261029
  ligand.dcd     : True
  production.csv : True
  hydration.csv  : True
  completed.json : True
  seed           : 20261029
  production_ps  : 10000.0
  frames         : 1000
  status         : completed

replica_02_seed20261030
  ligand.dcd     : True
  production.csv : True
  hydration.csv  : True
  completed.json : True
  seed           : 20261030
  production_ps  : 10000.0
  frames         : 1000
  status         : completed

replica_03_seed20261031
  ligand.dcd     : True
  production.csv : True
  hydration.csv  : True
  completed.json : True
  seed           : 20261031
  production_ps  : 10000.0
  frames         : 1000
  status         : completed

DO NOT RERUN MD.
09n OFFICIAL PRODUCTION RECOVERY: OK


## 2. Analisi conformazionale, idratazione e confronto descrittivo con 09l

Si scarta il primo **20%** di ciascuna produzione libera. L'RMSD conformazionale è calcolato dopo fit del ligando e non misura la ritenzione in un sito. Il raggio geometrico descrive l'estensione degli atomi pesanti. L'idratazione è il numero di ossigeni d'acqua entro 3,5 Å dagli atomi pesanti del ligando.

La traiettoria bound di riferimento è la produzione 09l MOL00484–EGFR da 1 ns, già verificata site-specificamente rispetto alla native EGF–EGFR interface. Le finestre temporali bound/free sono differenti; il confronto resta descrittivo e non costituisce una stima di energia libera o di entropia di binding.


In [7]:

print("=== 09n FREE-WATER CONFORMATIONAL ANALYSIS ===")

free_results = {}

for folder in replica_dirs:

    print(f"\n--- {folder.name} ---")

    traj = mt.load_dcd(
        str(folder / "ligand.dcd"),
        top=ligtop
    )

    require(traj.n_frames == 1000, "Attesi 1000 frame.")
    require(traj.n_atoms == 46, "Attesi 46 atomi.")
    require(np.isfinite(traj.xyz).all(), "Coordinate non finite.")

    # Primo frame della replica come riferimento conformazionale
    ref = traj[0]

    rmsd = mt.rmsd(
        traj,
        ref,
        atom_indices=heavy,
        ref_atom_indices=heavy
    ) * 10.0

    # Raggio geometrico degli atomi pesanti
    xyz = traj.xyz[:, heavy, :]
    center = xyz.mean(axis=1, keepdims=True)

    radius = np.sqrt(
        ((xyz - center) ** 2).sum(axis=2).mean(axis=1)
    ) * 10.0

    # Scarta il primo 20%
    discard = int(traj.n_frames * DISCARD_FRACTION)

    rmsd_analysis = rmsd[discard:]
    radius_analysis = radius[discard:]

    print("Analyzed frames :", len(rmsd_analysis))

    print(
        "RMSD mean ± SD  : "
        f"{rmsd_analysis.mean():.3f} ± "
        f"{rmsd_analysis.std(ddof=1):.3f} Å"
    )

    print(
        "RMSD min / max  : "
        f"{rmsd_analysis.min():.3f} / "
        f"{rmsd_analysis.max():.3f} Å"
    )

    print(
        "Radius mean ± SD: "
        f"{radius_analysis.mean():.3f} ± "
        f"{radius_analysis.std(ddof=1):.3f} Å"
    )

    free_results[folder.name] = {
        "rmsd": rmsd_analysis.copy(),
        "radius": radius_analysis.copy()
    }

    del traj


print("\nReplicas analyzed:", len(free_results))

require(
    len(free_results) == 3,
    "Non sono state analizzate tutte le tre repliche."
)

print("\nTEST E: OK")

=== TEST E — THREE FREE-WATER REPLICAS ===

--- replica_01_seed20261029 ---
Analyzed frames : 800
RMSD mean ± SD  : 1.874 ± 0.313 Å
RMSD min / max  : 0.508 / 2.501 Å
Radius mean ± SD: 4.707 ± 0.134 Å

--- replica_02_seed20261030 ---
Analyzed frames : 800
RMSD mean ± SD  : 1.120 ± 0.381 Å
RMSD min / max  : 0.258 / 2.286 Å
Radius mean ± SD: 4.704 ± 0.122 Å

--- replica_03_seed20261031 ---
Analyzed frames : 800
RMSD mean ± SD  : 1.314 ± 0.343 Å
RMSD min / max  : 0.377 / 2.316 Å
Radius mean ± SD: 4.708 ± 0.120 Å

Replicas analyzed: 3

TEST E: OK


In [8]:
print("=== 09n FREE-LIGAND HYDRATION ANALYSIS ===")

hydration_results = []

for folder in replica_dirs:

    print(f"\n--- {folder.name} ---")

    hydration_file = folder / "hydration.csv"

    require(
        hydration_file.exists(),
        f"File hydration mancante: {hydration_file}"
    )

    h = pd.read_csv(hydration_file)

    print("Rows:", len(h))
    print("Columns:", list(h.columns))

    require(
        "nearby_water_oxygens_3p5A" in h.columns,
        "Colonna hydration non trovata."
    )

    values = h["nearby_water_oxygens_3p5A"].to_numpy(float)

    require(
        np.isfinite(values).all(),
        "Valori hydration non finiti."
    )

    # discard first 20%
    discard = int(len(h) * DISCARD_FRACTION)
    values_analysis = values[discard:]

    print("Analyzed frames:", len(values_analysis))

    print(
        "Waters mean ± SD: "
        f"{values_analysis.mean():.3f} ± "
        f"{values_analysis.std(ddof=1):.3f}"
    )

    print(
        "Waters min / max: "
        f"{values_analysis.min():.0f} / "
        f"{values_analysis.max():.0f}"
    )

    hydration_results.append({
        "replica": folder.name,
        "mean": float(values_analysis.mean()),
        "sd": float(values_analysis.std(ddof=1)),
        "min": float(values_analysis.min()),
        "max": float(values_analysis.max())
    })


hydration_test = pd.DataFrame(hydration_results)

print("\n=== HYDRATION SUMMARY ===")
display(hydration_test)

require(
    len(hydration_test) == 3,
    "Non sono state analizzate tutte le repliche."
)

print("\nTEST G: OK")

=== TEST G — FREE-LIGAND HYDRATION ===

--- replica_01_seed20261029 ---
Rows: 1000
Columns: ['time_ps', 'nearby_water_oxygens_3p5A']
Analyzed frames: 800
Waters mean ± SD: 18.011 ± 3.058
Waters min / max: 9 / 27

--- replica_02_seed20261030 ---
Rows: 1000
Columns: ['time_ps', 'nearby_water_oxygens_3p5A']
Analyzed frames: 800
Waters mean ± SD: 18.000 ± 3.082
Waters min / max: 9 / 29

--- replica_03_seed20261031 ---
Rows: 1000
Columns: ['time_ps', 'nearby_water_oxygens_3p5A']
Analyzed frames: 800
Waters mean ± SD: 18.016 ± 3.110
Waters min / max: 9 / 28

=== HYDRATION SUMMARY ===


,replica,mean,sd,min,max
0,replica_01_seed20261029,18.01125,3.057828,9.0,27.0
1,replica_02_seed20261030,18.00000,3.081699,9.0,29.0
2,replica_03_seed20261031,18.01625,3.109755,9.0,28.0



TEST G: OK


In [13]:
print("=== 09n THERMODYNAMIC QC ===")

qc_rows = []

for folder in replica_dirs:

    print(f"\n--- {folder.name} ---")

    log_file = folder / "production.csv"

    require(
        log_file.exists(),
        f"production.csv mancante: {folder.name}"
    )

    log = pd.read_csv(log_file)

    print("Rows   :", len(log))
    print("Columns:", list(log.columns))

    temp_col = next(
        c for c in log.columns
        if "Temperature" in c
    )

    dens_col = next(
        c for c in log.columns
        if "Density" in c
    )

    temp = log[temp_col].to_numpy(float)
    density = log[dens_col].to_numpy(float)

    require(
        np.isfinite(temp).all(),
        f"Temperatura non finita: {folder.name}"
    )

    require(
        np.isfinite(density).all(),
        f"Densità non finita: {folder.name}"
    )

    mean_temp = float(temp.mean())
    sd_temp = float(temp.std(ddof=1))

    mean_density = float(density.mean())
    sd_density = float(density.std(ddof=1))

    temperature_OK = abs(
        mean_temp - TEMPERATURE_K
    ) < 5.0

    density_OK = (
        0.95 < mean_density < 1.05
    )

    qc_pass = bool(
        temperature_OK and density_OK
    )

    print(
        f"Temperature : "
        f"{mean_temp:.3f} ± {sd_temp:.3f} K"
    )

    print(
        f"Density     : "
        f"{mean_density:.5f} ± "
        f"{sd_density:.5f} g/mL"
    )

    print("Temperature QC:", temperature_OK)
    print("Density QC    :", density_OK)
    print("QC PASS       :", qc_pass)

    qc_rows.append({
        "replica": folder.name,
        "mean_temperature_K": mean_temp,
        "sd_temperature_K": sd_temp,
        "mean_density_g_mL": mean_density,
        "sd_density_g_mL": sd_density,
        "temperature_OK": temperature_OK,
        "density_OK": density_OK,
        "QC_PASS": qc_pass
    })


qc_test = pd.DataFrame(qc_rows)

print("\n=== QC SUMMARY ===")
display(qc_test)

require(
    len(qc_test) == 3,
    "Non sono state controllate tutte le repliche."
)

require(
    qc_test["QC_PASS"].all(),
    "Una o più repliche non superano il QC."
)

print("\nTEST H: OK")

=== TEST H — THERMODYNAMIC QC ===

--- replica_01_seed20261029 ---
Rows   : 1000
Columns: ['#"Step"', 'Time (ps)', 'Potential Energy (kJ/mole)', 'Kinetic Energy (kJ/mole)', 'Temperature (K)', 'Box Volume (nm^3)', 'Density (g/mL)', 'Speed (ns/day)']
Temperature : 299.978 ± 6.442 K
Density     : 0.99706 ± 0.01052 g/mL
Temperature QC: True
Density QC    : True
QC PASS       : True

--- replica_02_seed20261030 ---
Rows   : 1000
Columns: ['#"Step"', 'Time (ps)', 'Potential Energy (kJ/mole)', 'Kinetic Energy (kJ/mole)', 'Temperature (K)', 'Box Volume (nm^3)', 'Density (g/mL)', 'Speed (ns/day)']
Temperature : 300.455 ± 6.555 K
Density     : 0.99676 ± 0.00997 g/mL
Temperature QC: True
Density QC    : True
QC PASS       : True

--- replica_03_seed20261031 ---
Rows   : 1000
Columns: ['#"Step"', 'Time (ps)', 'Potential Energy (kJ/mole)', 'Kinetic Energy (kJ/mole)', 'Temperature (K)', 'Box Volume (nm^3)', 'Density (g/mL)', 'Speed (ns/day)']
Temperature : 300.347 ± 6.426 K
Density     : 0.99691 ± 0

,replica,mean_temperature_K,sd_temperature_K,mean_density_g_mL,sd_density_g_mL,temperature_OK,density_OK,QC_PASS
0,replica_01_seed20261029,299.977632,6.441934,0.997055,0.010520,True,True,True
1,replica_02_seed20261030,300.455421,6.554511,0.996761,0.009974,True,True,True
2,replica_03_seed20261031,300.346755,6.425781,0.996914,0.009965,True,True,True



TEST H: OK


In [12]:
# ============================================================
# 09n — SAFE CHUNKED LOAD OF 09l BOUND MOL00484
# Avoid full 9332-atom trajectory loading
# ============================================================

print("=== 09n SAFE BOUND TRAJECTORY LOAD ===")

# L è già definito dal preflight:
# 46 indici di MOL00484 nella traiettoria solute di 09l.

print("BOUND_DCD :", BOUND_DCD)
print("BOUND_TOP :", BOUND_TOPOLOGY)
print("Ligand atom indices :", len(L))

require(len(L) == 46, "Attesi 46 atomi MOL00484.")

bound_chunks = []

for i, chunk in enumerate(
    mt.iterload(
        str(BOUND_DCD),
        top=str(BOUND_TOPOLOGY),
        chunk=25,
        atom_indices=L
    )
):
    print(
        f"Chunk {i+1:02d} | "
        f"frames={chunk.n_frames} | "
        f"atoms={chunk.n_atoms}"
    )

    require(
        chunk.n_atoms == 46,
        "Il chunk non contiene esattamente MOL00484."
    )

    require(
        np.isfinite(chunk.xyz).all(),
        "Coordinate non finite nel chunk."
    )

    bound_chunks.append(chunk)


require(len(bound_chunks) > 0, "Nessun chunk caricato.")

bound = bound_chunks[0].join(bound_chunks[1:])

print("\n=== RESULT ===")
print("Bound frames :", bound.n_frames)
print("Bound atoms  :", bound.n_atoms)

require(
    bound.n_frames == 500,
    f"Attesi 500 frame 09l, trovati {bound.n_frames}"
)

require(
    bound.n_atoms == 46,
    f"Attesi 46 atomi MOL00484, trovati {bound.n_atoms}"
)

print(
    "Memory xyz  :",
    round(bound.xyz.nbytes / 1024**2, 3),
    "MB"
)

print("\n09n SAFE BOUND TRAJECTORY LOAD: OK")

=== 09n SAFE BOUND TRAJECTORY LOAD ===
BOUND_DCD : C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\production_runs\20261027T171745_698045Z_seed20261027\trajectories\production_solute.dcd
BOUND_TOP : C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\structures\solute_reference_09f.pdb
Ligand atom indices : 46
Chunk 01 | frames=25 | atoms=46
Chunk 02 | frames=25 | atoms=46
Chunk 03 | frames=25 | atoms=46
Chunk 04 | frames=25 | atoms=46
Chunk 05 | frames=25 | atoms=46
Chunk 06 | frames=25 | atoms=46
Chunk 07 | frames=25 | atoms=46
Chunk 08 | frames=25 | atoms=46
Chunk 09 | frames=25 | atoms=46
Chunk 10 | frames=25 | atoms=46
Chunk 11 | frames=25 | atoms=46
Chunk 12 | frames=25 | atoms=46
Chunk 13 | frames=25 | atoms=46
Chunk 14 | frames=25 | atoms=46
Chunk 15 | frames=25 | atoms=46
Chunk 16 | frames=25 | atoms=46
Chunk 17 | frames=25 | atoms=46
Chunk 18 | frames=25 | atoms=46
Chunk 19 | frames=25 | atoms=46
Chunk 20 | frames=25 | atoms=46

=== RESULT ===
Bound frames : 500
Bound atoms  : 46

In [4]:
print("=== 09n BOUND 09l CONFORMATIONAL REFERENCE ===")

require(bound.n_frames == 500, "Attesi 500 frame bound.")
require(bound.n_atoms == 46, "Attesi 46 atomi MOL00484.")
require(np.isfinite(bound.xyz).all(), "Coordinate bound non finite.")

print("Frames:", bound.n_frames)
print("Atoms :", bound.n_atoms)

# ------------------------------------------------------------
# Reference = primo frame della stessa traiettoria bound
# ------------------------------------------------------------

bound_ref = bound[0]

print("Reference frames:", bound_ref.n_frames)
print("Reference atoms :", bound_ref.n_atoms)

# ------------------------------------------------------------
# Heavy-atom self-fit conformational RMSD
# ------------------------------------------------------------

bound_rmsd = mt.rmsd(
    bound,
    bound_ref,
    atom_indices=heavy,
    ref_atom_indices=heavy
) * 10.0

require(
    np.isfinite(bound_rmsd).all(),
    "RMSD bound contiene valori non finiti."
)

# ------------------------------------------------------------
# Heavy-atom geometric radius
# ------------------------------------------------------------

xyz = bound.xyz[:, heavy, :]

center = xyz.mean(
    axis=1,
    keepdims=True
)

bound_radius = np.sqrt(
    ((xyz - center) ** 2)
    .sum(axis=2)
    .mean(axis=1)
) * 10.0

require(
    np.isfinite(bound_radius).all(),
    "Raggio bound contiene valori non finiti."
)

# ------------------------------------------------------------
# Same 20% discard used in 09l/09n analysis
# ------------------------------------------------------------

discard = int(
    bound.n_frames * DISCARD_FRACTION
)

bound_rmsd_analysis = bound_rmsd[discard:]
bound_radius_analysis = bound_radius[discard:]

print("\nAnalyzed frames :", len(bound_rmsd_analysis))

print(
    "RMSD mean ± SD  : "
    f"{bound_rmsd_analysis.mean():.3f} ± "
    f"{bound_rmsd_analysis.std(ddof=1):.3f} Å"
)

print(
    "RMSD min / max  : "
    f"{bound_rmsd_analysis.min():.3f} / "
    f"{bound_rmsd_analysis.max():.3f} Å"
)

print(
    "Radius mean ± SD: "
    f"{bound_radius_analysis.mean():.3f} ± "
    f"{bound_radius_analysis.std(ddof=1):.3f} Å"
)

print(
    "Radius min / max: "
    f"{bound_radius_analysis.min():.3f} / "
    f"{bound_radius_analysis.max():.3f} Å"
)

print("\nTEST F: OK")

=== TEST F — BOUND MOL00484 CONFORMATION ===
Frames: 500
Atoms : 46
Reference frames: 1
Reference atoms : 46

Analyzed frames : 400
RMSD mean ± SD  : 1.142 ± 0.213 Å
RMSD min / max  : 0.693 / 1.696 Å
Radius mean ± SD: 4.729 ± 0.079 Å
Radius min / max: 4.477 / 4.958 Å

TEST F: OK


In [16]:
print("=== 09n — SAVE OFFICIAL NUMERICAL RESULTS ===")

# ------------------------------------------------------------
# Free conformational summary
# ------------------------------------------------------------

rows = []

for replica, values in free_results.items():

    rmsd = values["rmsd"]
    radius = values["radius"]

    rows.append({
        "state": "free",
        "series": replica,
        "analyzed_frames": len(rmsd),

        "RMSD_mean_A": float(rmsd.mean()),
        "RMSD_SD_A": float(rmsd.std(ddof=1)),
        "RMSD_min_A": float(rmsd.min()),
        "RMSD_max_A": float(rmsd.max()),

        "radius_mean_A": float(radius.mean()),
        "radius_SD_A": float(radius.std(ddof=1)),
        "radius_min_A": float(radius.min()),
        "radius_max_A": float(radius.max())
    })


# ------------------------------------------------------------
# Bound conformational summary
# Values independently verified in TEST F
# 400 frames after 20% discard
# ------------------------------------------------------------

rows.append({
    "state": "bound",
    "series": "bound_09l_1ns",
    "analyzed_frames": 400,

    "RMSD_mean_A": 1.142,
    "RMSD_SD_A": 0.213,
    "RMSD_min_A": 0.693,
    "RMSD_max_A": 1.696,

    "radius_mean_A": 4.729,
    "radius_SD_A": 0.079,
    "radius_min_A": 4.477,
    "radius_max_A": 4.958
})


conformation_summary = pd.DataFrame(rows)

conformation_summary.to_csv(
    RUN / "conformation_summary.csv",
    index=False
)


# ------------------------------------------------------------
# Hydration
# ------------------------------------------------------------

hydration_test.to_csv(
    RUN / "hydration_summary.csv",
    index=False
)


# ------------------------------------------------------------
# Thermodynamic QC
# ------------------------------------------------------------

qc_test.to_csv(
    RUN / "09n_QC.csv",
    index=False
)


# ------------------------------------------------------------
# Final JSON summary
# ------------------------------------------------------------

summary = {
    "candidate": "MOL00484",

    "analysis": "09n_free_water_MD_comparison",

    "free_water_replicas": 3,
    "production_ns_per_replica": 10.0,
    "total_free_water_production_ns": 30.0,

    "discard_fraction": 0.20,

    "microstate":
        "neutral_same_as_authoritative_09l",

    "rmsd_definition": (
        "heavy-atom self-fit conformational RMSD "
        "relative to the first frame of each trajectory"
    ),

    "radius_definition":
        "heavy-atom geometric radius",

    "hydration_definition": (
        "number of water oxygen atoms within "
        "3.5 Angstrom of ligand heavy atoms"
    ),

    "technical_qc_pass":
        bool(qc_test["QC_PASS"].all()),

    "bound_reference":
        "MOL00484 09l 1-ns site-relevant EGFR trajectory",

    "bound_site_context":
        "native EGF-EGFR interface site-relevant",

    "important_limitation": (
        "Self-referenced RMSD values quantify internal "
        "conformational deviation within each trajectory "
        "and are not absolute structural distances "
        "between free and bound states."
    ),

    "interpretation": (
        "Descriptive conformational and hydration comparison "
        "at fixed microstate. This analysis does not estimate "
        "binding free energy, entropy, Kd, Ki, or inhibition."
    ),

    "experimentally_validated": False
}

dump(
    RUN / "09n_summary.json",
    summary
)


print("\n=== CONFORMATION ===")
display(conformation_summary)

print("\n=== HYDRATION ===")
display(hydration_test)

print("\n=== QC ===")
display(qc_test)

print("\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))

print("\nOutput:", RUN)

print("\n09n OFFICIAL NUMERICAL RESULTS: SAVED")

=== 09n — SAVE OFFICIAL NUMERICAL RESULTS ===

=== CONFORMATION ===


,state,series,analyzed_frames,RMSD_mean_A,RMSD_SD_A,RMSD_min_A,RMSD_max_A,radius_mean_A,radius_SD_A,radius_min_A,radius_max_A
0,free,replica_01_seed20261029,800,1.874125,0.312686,0.508267,2.500700,4.707225,0.134492,4.247385,5.034321
1,free,replica_02_seed20261030,800,1.120270,0.381060,0.257716,2.286381,4.703785,0.121603,4.316650,5.034552
2,free,replica_03_seed20261031,800,1.314390,0.342885,0.377241,2.315526,4.707919,0.120046,4.342043,5.043244
3,bound,bound_09l_1ns,400,1.142000,0.213000,0.693000,1.696000,4.729000,0.079000,4.477000,4.958000



=== HYDRATION ===


,replica,mean,sd,min,max
0,replica_01_seed20261029,18.01125,3.057828,9.0,27.0
1,replica_02_seed20261030,18.00000,3.081699,9.0,29.0
2,replica_03_seed20261031,18.01625,3.109755,9.0,28.0



=== QC ===


,replica,mean_temperature_K,sd_temperature_K,mean_density_g_mL,sd_density_g_mL,temperature_OK,density_OK,QC_PASS
0,replica_01_seed20261029,299.977632,6.441934,0.997055,0.010520,True,True,True
1,replica_02_seed20261030,300.455421,6.554511,0.996761,0.009974,True,True,True
2,replica_03_seed20261031,300.346755,6.425781,0.996914,0.009965,True,True,True



=== SUMMARY ===
{
  "candidate": "MOL00484",
  "analysis": "09n_free_water_MD_comparison",
  "free_water_replicas": 3,
  "production_ns_per_replica": 10.0,
  "total_free_water_production_ns": 30.0,
  "discard_fraction": 0.2,
  "microstate": "neutral_same_as_authoritative_09l",
  "rmsd_definition": "heavy-atom self-fit conformational RMSD relative to the first frame of each trajectory",
  "radius_definition": "heavy-atom geometric radius",
  "hydration_definition": "number of water oxygen atoms within 3.5 Angstrom of ligand heavy atoms",
  "technical_qc_pass": true,
  "bound_reference": "MOL00484 09l 1-ns site-relevant EGFR trajectory",
  "bound_site_context": "native EGF-EGFR interface site-relevant",
  "important_limitation": "Self-referenced RMSD values quantify internal conformational deviation within each trajectory and are not absolute structural distances between free and bound states.",
  "interpretation": "Descriptive conformational and hydration comparison at fixed microstate

## Interpretazione e passaggio successivo

09n caratterizza MOL00484 **libero in acqua** e lo confronta descrittivamente con la configurazione MOL00484–EGFR simulata in 09l. I risultati non devono essere trasformati in stime di ΔG, Kd/Ki, efficacia o inibizione.

Dopo il completamento e il QC delle 3 × 10 ns libere, il passaggio successivo è **09o — MOL00484–EGFR bound CUDA replicates**, con 3 repliche × 10 ns e verifica site-specifica della native EGF–EGFR interface.


In [1]:
from pathlib import Path

BASE = Path(
    r"C:\Users\Roberto\PEARL\outputs"
    r"\pipeline_5_09n_mol00484_free_water"
)

print("BASE exists:", BASE.exists())
print("BASE:", BASE)

if BASE.exists():

    print("\n" + "=" * 80)
    print("09n DIRECTORY CONTENT")
    print("=" * 80)

    for p in sorted(BASE.rglob("*")):

        if p.is_file():

            rel = p.relative_to(BASE)

            size_kb = p.stat().st_size / 1024

            print(f"{str(rel):80s} {size_kb:10.1f} KB")

BASE exists: True
BASE: C:\Users\Roberto\PEARL\outputs\pipeline_5_09n_mol00484_free_water

09n DIRECTORY CONTENT
20261028T071151_079899Z\provenance.json                                                 0.6 KB
20261028T071151_079899Z\replica_01_seed20261029\ligand_template.xml                    11.0 KB
20261028T071403_207257Z\provenance.json                                                 0.5 KB
20261028T071432_863077Z\provenance.json                                                 0.6 KB
20261028T071432_863077Z\replica_01_seed20261029\ligand_template.xml                    11.0 KB
20261028T071432_863077Z\replica_01_seed20261029\minimized.chk                        1082.7 KB
20261028T071432_863077Z\replica_01_seed20261029\minimized.xml                         242.8 KB
20261028T071432_863077Z\replica_01_seed20261029\nvt.csv                                 0.1 KB
20261028T071432_863077Z\replica_01_seed20261029\solvated.pdb                          175.1 KB
20261028T080153_298689Z_smoke\fi

In [2]:
from pathlib import Path
import pandas as pd
import json

RUN = Path(
    r"C:\Users\Roberto\PEARL\outputs"
    r"\pipeline_5_09n_mol00484_free_water"
    r"\20261028T080940_432915Z_production"
)

print("=" * 80)
print("09n PRODUCTION RUN")
print("=" * 80)
print(RUN)

# ------------------------------------------------------------
# Summary files
# ------------------------------------------------------------

for name in [
    "conformation_summary.csv",
    "hydration_summary.csv",
    "09n_QC.csv",
]:

    path = RUN / name

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:")
    print(list(df.columns))

    print("\nContent:")
    print(df.to_string(index=False))


# ------------------------------------------------------------
# Per-replica hydration files
# ------------------------------------------------------------

replicas = [
    "replica_01_seed20261029",
    "replica_02_seed20261030",
    "replica_03_seed20261031",
]

for replica in replicas:

    path = RUN / replica / "hydration.csv"

    print("\n" + "=" * 80)
    print(replica, "- hydration.csv")
    print("=" * 80)

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:")
    print(list(df.columns))

    print("\nFirst 5 rows:")
    print(df.head().to_string(index=False))

    print("\nLast 5 rows:")
    print(df.tail().to_string(index=False))


# ------------------------------------------------------------
# Summary JSON
# ------------------------------------------------------------

summary_json = RUN / "09n_summary.json"

print("\n" + "=" * 80)
print("09n_summary.json")
print("=" * 80)

with open(summary_json, "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data, indent=2))

09n PRODUCTION RUN
C:\Users\Roberto\PEARL\outputs\pipeline_5_09n_mol00484_free_water\20261028T080940_432915Z_production

conformation_summary.csv
Shape: (4, 11)
Columns:
['state', 'series', 'analyzed_frames', 'RMSD_mean_A', 'RMSD_SD_A', 'RMSD_min_A', 'RMSD_max_A', 'radius_mean_A', 'radius_SD_A', 'radius_min_A', 'radius_max_A']

Content:
state                  series  analyzed_frames  RMSD_mean_A  RMSD_SD_A  RMSD_min_A  RMSD_max_A  radius_mean_A  radius_SD_A  radius_min_A  radius_max_A
 free replica_01_seed20261029              800     1.874125   0.312686    0.508267    2.500700       4.707225     0.134492      4.247385      5.034321
 free replica_02_seed20261030              800     1.120270   0.381060    0.257716    2.286381       4.703785     0.121603      4.316650      5.034552
 free replica_03_seed20261031              800     1.314390   0.342885    0.377241    2.315526       4.707919     0.120046      4.342043      5.043244
bound           bound_09l_1ns              400     1.1420